# Parse PDFs with PyMuPDF

This notebook recursively finds PDF files inside `data/` (across subfolders such as `ero-reliability-risk-priorities-reports/`, `event_analysis_reports/`, and `state_of_reliability_reports/`), parses them with the open-source `PyMuPDF` package, and saves the extracted text as JSON.

Each PDF's parent folder name is captured as its `category`, so downstream steps can filter or group by report type.

## Setup

Install the dependencies from `requirements.txt` before running the notebook:

```bash
pip install -r requirements.txt
```

Expected layout (PDFs live in category subfolders under `data/`):

```
data_pipeline/
└── data/
    ├── ero-reliability-risk-priorities-reports/*.pdf
    ├── event_analysis_reports/*.pdf
    └── state_of_reliability_reports/*.pdf
```

The notebook works whether you launch Jupyter from the repo root or from inside `data_pipeline/`.

In [1]:
from pathlib import Path
import json

import pandas as pd
from tqdm.auto import tqdm
import fitz  # PyMuPDF

/opt/homebrew/Caskroom/miniconda/base/envs/gridsync/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
CANDIDATE_DATA_DIRS = [Path("data_pipeline/data"), Path("data")]
DATA_DIR = next((p for p in CANDIDATE_DATA_DIRS if p.exists()), CANDIDATE_DATA_DIRS[0])

CANDIDATE_OUTPUT_DIRS = [Path("data_pipeline"), Path(".")]
OUTPUT_DIR = next((p for p in CANDIDATE_OUTPUT_DIRS if p.exists()), CANDIDATE_OUTPUT_DIRS[-1])
OUTPUT_PATH = OUTPUT_DIR / "parsed_pdfs.json"

DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f"Reading PDFs from: {DATA_DIR.resolve()}")
print(f"Saving parsed output to: {OUTPUT_PATH.resolve()}")

categories = sorted(p.name for p in DATA_DIR.iterdir() if p.is_dir())
print(f"Found {len(categories)} category folder(s): {categories}")

Reading PDFs from: /Users/ayaanehsan/Developer/scsp/GridSync/data_pipeline/data
Saving parsed output to: /Users/ayaanehsan/Developer/scsp/GridSync/data_pipeline/parsed_pdfs.json
Found 3 category folder(s): ['ero-reliability-risk-priorities-reports', 'event_analysis_reports', 'state_of_reliability_reports']


In [3]:
pdf_files = sorted(DATA_DIR.rglob("*.pdf"))
print(f"Found {len(pdf_files)} PDF file(s) across all subfolders.\n")

from collections import Counter

counts_by_category = Counter(
    pdf.relative_to(DATA_DIR).parts[0] if pdf.relative_to(DATA_DIR).parts[:-1] else "(root)"
    for pdf in pdf_files
)
for category, count in sorted(counts_by_category.items()):
    print(f"  {category}: {count} PDF(s)")

Found 64 PDF file(s) across all subfolders.

  ero-reliability-risk-priorities-reports: 12 PDF(s)
  event_analysis_reports: 33 PDF(s)
  state_of_reliability_reports: 19 PDF(s)


In [ ]:
# I want to loop over pdf files in event_analysis_reports and convert each pdf to text and then parse it to the Vector database (you can create a dummy vector database parsing function)
